# Colab Streamlit Demo: Routed Qwen2.5-VL QA

Use this notebook when you want Colab GPU to host the Streamlit demo.

Flow:

```text
Colab GPU
-> clone/pull repo
-> install dependencies
-> copy router/adapters from Drive if needed
-> start Streamlit
-> expose Streamlit URL with cloudflared
```

The Streamlit app uses the project pipeline:

```text
image + question -> multimodal router -> selected Qwen backend -> answer
```

## 1. Check GPU

In [ ]:
import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("cuda version:", torch.version.cuda)
else:
    print("Runtime > Change runtime type > GPU")

## 2. Clone Or Pull Repo

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git"
PROJECT_ROOT = Path("/content/multi-task-moe-vlm-assistant")
BRANCH = "main"


def run(cmd, check=True):
    print("Running:", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd)
    return result


if PROJECT_ROOT.exists() and (PROJECT_ROOT / ".git").exists():
    run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", BRANCH])
    run(["git", "-C", str(PROJECT_ROOT), "reset", "--hard", f"origin/{BRANCH}"])
elif PROJECT_ROOT.exists():
    raise RuntimeError(f"{PROJECT_ROOT} exists but is not a git repo. Remove it or choose another path.")
else:
    run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_ROOT)])

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
run(["git", "-C", str(PROJECT_ROOT), "log", "-1", "--oneline"])

## 3. Install Dependencies

This can take a few minutes. Restarting the runtime after large installs is sometimes helpful, but usually not required.

In [ ]:
%cd /content/multi-task-moe-vlm-assistant
%pip install -q -r requirements.txt
%pip install -q -U qwen-vl-utils python-multipart

## 4. Optional: Hugging Face Login

Use this if model downloads are slow/rate-limited or if your model access requires authentication.

In [ ]:
# Optional. Uncomment if needed.
# from huggingface_hub import login
# login()

## 5. Mount Drive And Copy Checkpoints

Edit `DRIVE_ROOT` if your checkpoint folder is somewhere else.

Expected local paths after copy:

```text
checkpoints/router/multimodal_deberta_clip_router/multimodal_logreg.joblib
checkpoints/chart_dora_r8_a16_B_lr2e-5/chart_dora_r8_a16_B_lr2e-5/adapter_config.json
checkpoints/textvqa_lora/textvqa_lora/adapter_config.json
```

In [ ]:
from pathlib import Path
import shutil

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/multi-task-moe-vlm-assistant")
DRIVE_ROOT = Path("/content/drive/MyDrive/multi-task-moe-vlm-assistant/checkpoints")

COPY_TARGETS = [
    (
        DRIVE_ROOT / "router/multimodal_deberta_clip_router",
        PROJECT_ROOT / "checkpoints/router/multimodal_deberta_clip_router",
    ),
    (
        DRIVE_ROOT / "chart_dora_r8_a16_B_lr2e-5",
        PROJECT_ROOT / "checkpoints/chart_dora_r8_a16_B_lr2e-5",
    ),
    (
        DRIVE_ROOT / "textvqa_lora",
        PROJECT_ROOT / "checkpoints/textvqa_lora",
    ),
]

for src, dst in COPY_TARGETS:
    print("source:", src)
    print("target:", dst)
    if not src.exists():
        print("  missing source, skipped")
        continue
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    print("  copied")

checks = [
    PROJECT_ROOT / "checkpoints/router/multimodal_deberta_clip_router/multimodal_logreg.joblib",
    PROJECT_ROOT / "checkpoints/chart_dora_r8_a16_B_lr2e-5/chart_dora_r8_a16_B_lr2e-5/adapter_config.json",
    PROJECT_ROOT / "checkpoints/textvqa_lora/textvqa_lora/adapter_config.json",
]
print("\nCheckpoint checks:")
for path in checks:
    print(path, "exists=", path.exists())

## 6. Pre-download Qwen2.5-VL 7B Optional

The first answer request will download/load Qwen if it is not cached. This cell lets you do that before launching the app.

In [ ]:
# Optional. This downloads the model into the Colab/HF cache first.
# It can take several minutes and needs enough disk space.
# from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
# MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
# AutoProcessor.from_pretrained(MODEL_NAME)
# Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_NAME, device_map="auto")
# print("Qwen cached/loaded")

## 7. Start Streamlit In Background

This starts the demo app on Colab localhost port `8501`.

In [ ]:
%cd /content/multi-task-moe-vlm-assistant
!pkill -f streamlit || true
!nohup streamlit run scripts/serve_streamlit.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &
!sleep 5
!tail -n 40 streamlit.log

## 8. Open Public Tunnel

Run this cell and copy the `https://...trycloudflare.com` URL. Keep the cell running while you use the demo.

In [ ]:
%cd /content/multi-task-moe-vlm-assistant
!if [ ! -f /content/cloudflared ]; then wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared; fi
!/content/cloudflared tunnel --url http://localhost:8501


## 9. Debug Logs

Use these if the app page opens but answer generation fails.

In [ ]:
%cd /content/multi-task-moe-vlm-assistant
!tail -n 120 streamlit.log